# 🤖 Automated Freqtrade Backtest Runner Node (Google Colab Cloud)
### Closed-Loop Agentic Optimization Compute Node

**Automated Workflow:**
1. 📥 Pull candidate strategy from GitHub repository (`tminhnhat/NostalgiaForInfinity`).
2. ⚡ Sync candles directly from Cloudflare R2 bucket (`freqtrade-data/`) to Colab local NVMe.
3. 🏎️ Execute multi-threaded Freqtrade backtest across rolling evaluation window.
4. 📊 Run `scripts/export_report.py` to generate `reports/history/backtest_<date>.json` and `reports/latest_report.md`.
5. 🚀 Auto-commit and push report artifacts to branch `eval/daily-report` to trigger the Jules Optimizer Agent.

## ⚙️ 1. Environment Configuration & Secrets Setup

In [ ]:
import os

# Set credentials (can also be loaded via google.colab.userdata or environment variables)
try:
    from google.colab import userdata
    GH_PAT = userdata.get('GH_PAT')
    R2_ENDPOINT = userdata.get('R2_ENDPOINT') or 'https://a52481108c8613c521b7670d797d8848.r2.cloudflarestorage.com'
    R2_ACCESS_KEY = userdata.get('R2_ACCESS_KEY') or userdata.get('R2_ACCESS_KEY_ID') or '0da5c6eb0ad37511c20061f4217197f0'
    R2_SECRET_KEY = userdata.get('R2_SECRET_KEY') or userdata.get('R2_SECRET_ACCESS_KEY') or 'aeb545c076f75b051be1128efd91fe7e47bd6d042fe9577e3b19fb0bf6e218c4'
    R2_BUCKET = userdata.get('R2_BUCKET') or userdata.get('R2_BUCKET_NAME') or 'goclaw-data'
    R2_REGION = userdata.get('R2_REGION') or 'auto'
except Exception:
    GH_PAT = os.environ.get('GH_PAT', '')
    R2_ENDPOINT = os.environ.get('R2_ENDPOINT', 'https://a52481108c8613c521b7670d797d8848.r2.cloudflarestorage.com')
    R2_ACCESS_KEY = os.environ.get('R2_ACCESS_KEY') or os.environ.get('R2_ACCESS_KEY_ID', '0da5c6eb0ad37511c20061f4217197f0')
    R2_SECRET_KEY = os.environ.get('R2_SECRET_KEY') or os.environ.get('R2_SECRET_ACCESS_KEY', 'aeb545c076f75b051be1128efd91fe7e47bd6d042fe9577e3b19fb0bf6e218c4')
    R2_BUCKET = os.environ.get('R2_BUCKET') or os.environ.get('R2_BUCKET_NAME', 'goclaw-data')
    R2_REGION = os.environ.get('R2_REGION', 'auto')

REPO_URL = f'https://{GH_PAT}@github.com/tminhnhat/NostalgiaForInfinity.git' if GH_PAT else 'https://github.com/tminhnhat/NostalgiaForInfinity.git'
WORKDIR = '/content/botx8'
STRATEGY_NAME = 'NostalgiaForInfinityX8'
REPORT_BRANCH = 'eval/daily-report'

print('✅ Configuration loaded successfully.')


## 📦 2. Install TA-Lib C, Dependencies, and Rclone

In [ ]:
# Install Rclone
!apt-get update -qq && apt-get install -y -qq rclone > /dev/null

# Install TA-Lib C Library
!wget -q http://prdownloads.sourceforge.net/ta-lib/ta-lib-0.4.0-src.tar.gz
!tar -xzf ta-lib-0.4.0-src.tar.gz
%cd ta-lib/
!./configure --prefix=/usr > /dev/null
!make -j$(nproc) > /dev/null
!make install > /dev/null
%cd ..
!rm -rf ta-lib ta-lib-0.4.0-src.tar.gz

# Install Python packages
!pip install -q TA-Lib freqtrade pyarrow fastparquet pandas

!freqtrade --version

## 🔄 3. Clone Repository & Setup Cloudflare R2 Connection

In [ ]:
import subprocess

# Clone or pull repository
if not os.path.exists(WORKDIR):
    !git clone $REPO_URL $WORKDIR
%cd $WORKDIR
!git checkout main
!git pull origin main

# Configure Rclone for Cloudflare R2
rclone_conf = f"""[r2]
type = s3
provider = Cloudflare
access_key_id = {R2_ACCESS_KEY}
secret_access_key = {R2_SECRET_KEY}
endpoint = {R2_ENDPOINT}
region = {R2_REGION}
acl = private
"""
os.makedirs(os.path.expanduser('~/.config/rclone'), exist_ok=True)
with open(os.path.expanduser('~/.config/rclone/rclone.conf'), 'w') as f:
    f.write(rclone_conf)

print('✅ Rclone configured for Cloudflare R2 endpoint:', R2_ENDPOINT)


## 📥 4. Sync Target Candles from Cloudflare R2

In [ ]:
# Sync candles directly from R2 to local NVMe SSD
os.makedirs('user_data/data/binance', exist_ok=True)
!rclone sync r2:{R2_BUCKET}/binance user_data/data/binance --fast-list --transfers=16 -v

parquet_count = len(list(os.scandir('user_data/data/binance')))
print(f'✅ Sync completed! Found {parquet_count} candle files in local storage.')

## 🚀 5. Execute Freqtrade Rolling Window Backtest

In [ ]:
from datetime import datetime, timedelta, timezone

# Compute rolling 30-day evaluation window
now = datetime.now(timezone.utc)
start_date = (now - timedelta(days=30)).strftime('%Y%m%d')
end_date = now.strftime('%Y%m%d')
timerange = f"{start_date}-{end_date}"
print(f'🎯 Evaluation Window: {timerange}')

!freqtrade backtesting \
  --config configs/exampleconfig.json \
  --config configs/pairlist-backtest-static-binance-spot-usdt.json \
  --strategy {STRATEGY_NAME} \
  --strategy-path . \
  --timerange {timerange} \
  --timeframe 5m \
  --export trades \
  --breakdown day

## 📊 6. Generate Metrics Report Artifacts

In [ ]:
# Run report export script to produce backtest_<YYYYMMDD>.json and latest_report.md
!python scripts/export_report.py --strategy {STRATEGY_NAME}

# Display markdown report in notebook
from IPython.display import display, Markdown
with open('reports/latest_report.md', 'r', encoding='utf-8') as f:
    display(Markdown(f.read()))

## 📤 7. Commit & Push to GitHub Branch `eval/daily-report`

In [ ]:
# Fetch or switch to eval/daily-report branch and push updates
!git config user.name "Colab Evaluation Node"
!git config user.email "actions@github.com"

!git checkout -B eval/daily-report
!git add reports/history/ reports/latest_report.md
!git commit -m "chore(eval): automated daily backtest report [$(date -u +'%Y-%m-%d')]"
!git push -u origin eval/daily-report --force

print('✅ Daily evaluation report published to branch eval/daily-report!')
print('🚀 Jules Optimizer Agent will be triggered automatically.')